# Field Extraction: Cytiva "Animal Origin Statement" (PyMuPDF + OCR)
**Document:** `pharmaceutical-sdf-page2-animal-origin.pdf` — a single-page Cytiva Animal Origin Statement
for ÄKTA™ ready products.
**Author:**

Like `Sample Documentation.pdf`, this page has **zero words** via `page.get_text("words")` — it's a fully
rasterized scan, so the entire pipeline runs through Tesseract OCR on the rendered page image.

This notebook:
1. Confirms there is no text layer.
2. OCRs the page and extracts 8 labeled fields (basic info + dates + a regulatory reference).
3. Extracts the Product/Part-name table as a structured list.
4. Flags two genuinely interesting OCR-specific findings: a rotated approval stamp that OCR *can* recover
   (unlike native text extraction), and a printed "Name: Susan Boucher" field that OCR completely misses.
5. Draws OpenCV rectangles over the page to visually confirm every box.


## Setup

In [ ]:
# In Colab, uncomment the line below first:
# !apt-get -qq install tesseract-ocr -y && pip install pymupdf pytesseract opencv-python-headless -q

import json
import pymupdf
import pytesseract
from pytesseract import Output
import numpy as np
import cv2
from google.colab import files  # noqa: F401  (Colab only; comment out if running locally)


## Upload the PDF

In [ ]:
try:
    uploaded = files.upload()
    SRC = list(uploaded.keys())[0]
except Exception:
    SRC = "pharmaceutical-sdf-page2-animal-origin.pdf"
print("Using file:", SRC)


## Step 0 — confirm there is no text layer

In [ ]:
doc = pymupdf.open(SRC)
print(f"{len(doc)} page(s)")
print("get_text('words') on page 1:", len(doc[0].get_text("words")), "words")


## 8 fields identified for extraction
| # | Field | Type |
|---|---|---|
| 1 | Document Title ("Animal Origin Statement") | Basic info |
| 2 | Regarding / Subject | Basic info |
| 3 | Vendor Name (Cytiva) | Basic info |
| 4 | Regulatory Standard Referenced (EMA 410/01 Rev 3) | Basic info |
| 5 | Signature Date | Date field |
| 6 | Signee Title | Basic info |
| 7 | Document Number (DOC1098513/Rev9) | Basic info |
| 8 | Rotated Approval Stamp timestamp | Date field (bonus — see note below) |


## OCR + label-anchored extraction helpers

In [ ]:
"""
OCR-based field & table extraction for pharmaceutical-sdf-page2-animal-origin.pdf
(a single-page Cytiva "Animal Origin Statement" for AKTA ready products).

Like the other Pfizer-folder document, page.get_text("words") returns ZERO words --
this is a fully rasterized/scanned page with no text layer at all, so every field
has to come from OCR (Tesseract) on the rendered page image.
"""
import json
import pymupdf
import pytesseract
from pytesseract import Output
import numpy as np
import cv2

SRC = "pharmaceutical-sdf-page2-animal-origin.pdf"
ZOOM = 3.0
Y_TOL_PX = 12


def render_page(doc, page_index, zoom=ZOOM):
    page = doc[page_index]
    pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom))
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    return cv2.cvtColor(img, cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR)


def ocr_words(img, conf_thresh=40):
    data = pytesseract.image_to_data(img, output_type=Output.DICT)
    words = []
    for i in range(len(data["text"])):
        text = data["text"][i].strip()
        conf = float(data["conf"][i])
        if not text or conf < conf_thresh:
            continue
        x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
        words.append((x, y, x + w, y + h, text))
    return words


def to_pdf_coords(bbox_px, zoom=ZOOM):
    return [round(v / zoom, 1) for v in bbox_px]


def union_bbox(words):
    return (min(w[0] for w in words), min(w[1] for w in words),
            max(w[2] for w in words), max(w[3] for w in words))


def find_label(words, label_tokens):
    norm = lambda s: s.strip(".:()").lower()
    n = len(label_tokens)
    for i in range(len(words) - n + 1):
        chunk = words[i:i + n]
        if all(norm(chunk[j][4]) == norm(label_tokens[j]) for j in range(n)):
            return chunk
    return None


def value_to_right(words, label_chunk, max_gap_px=700, y_tol=Y_TOL_PX):
    lx1 = label_chunk[-1][2]
    ly_mid = (label_chunk[-1][1] + label_chunk[-1][3]) / 2
    cands = [w for w in words if abs((w[1] + w[3]) / 2 - ly_mid) <= y_tol
             and w[0] > lx1 and w[0] - lx1 < max_gap_px]
    cands.sort(key=lambda w: w[0])
    return cands


def value_below(words, label_chunk, max_dy=60, x_tol=350):
    lx0 = label_chunk[0][0]
    ly1 = label_chunk[0][3]
    same_col = [w for w in words if w[1] > ly1 and w[1] - ly1 < max_dy and abs(w[0] - lx0) < x_tol]
    if not same_col:
        return []
    row_y = min(w[1] for w in same_col)
    row = [w for w in same_col if abs(w[1] - row_y) <= Y_TOL_PX]
    row.sort(key=lambda w: w[0])
    return row


def cluster_rows(words, y_tol=Y_TOL_PX):
    rows, used = [], [False] * len(words)
    order = sorted(range(len(words)), key=lambda i: (words[i][1] + words[i][3]) / 2)
    for idx in order:
        if used[idx]:
            continue
        base_y = (words[idx][1] + words[idx][3]) / 2
        row = [words[idx]]
        used[idx] = True
        for j in order:
            if not used[j] and abs((words[j][1] + words[j][3]) / 2 - base_y) <= y_tol:
                row.append(words[j])
                used[j] = True
        row.sort(key=lambda w: w[0])
        rows.append(row)
    rows.sort(key=lambda r: r[0][1])
    return rows


def extract_fields(words):
    fields = []

    title_chunk = find_label(words, ["Animal", "Origin", "Statement"])
    if title_chunk:
        fields.append({"field": "Document Title", "text": "Animal Origin Statement",
                        "bbox_px": union_bbox(title_chunk), "bbox_pt": to_pdf_coords(union_bbox(title_chunk))})

    regarding_chunk = find_label(words, ["Regarding:"])
    if regarding_chunk:
        val = value_to_right(words, regarding_chunk, max_gap_px=1200)
        if val:
            fields.append({"field": "Regarding / Subject", "text": " ".join(w[4] for w in val),
                            "bbox_px": union_bbox(val), "bbox_pt": to_pdf_coords(union_bbox(val))})

    # Vendor name: "cytiva" logo wordmark, top-left
    for w in words:
        if w[4].lower() == "cytiva" and w[1] < 300:
            fields.append({"field": "Vendor Name", "text": "Cytiva",
                            "bbox_px": (w[0], w[1], w[2], w[3]), "bbox_pt": to_pdf_coords((w[0], w[1], w[2], w[3]))})
            break

    # Regulatory reference standard (EMA 410/01 Rev 3) inside the statement paragraph
    ema_chunk = find_label(words, ["EMA"])
    if ema_chunk:
        val = value_to_right(words, ema_chunk, max_gap_px=300)
        if val:
            full = ema_chunk + val
            fields.append({"field": "Regulatory Standard Referenced", "text": " ".join(w[4] for w in full),
                            "bbox_px": union_bbox(full), "bbox_pt": to_pdf_coords(union_bbox(full))})

    date_chunk = find_label(words, ["Date:"])
    if date_chunk:
        val = value_below(words, date_chunk)
        if val:
            fields.append({"field": "Signature Date", "text": " ".join(w[4] for w in val),
                            "bbox_px": union_bbox(val), "bbox_pt": to_pdf_coords(union_bbox(val))})

    title_role_chunk = find_label(words, ["Title"])
    if title_role_chunk:
        val = value_below(words, title_role_chunk, max_dy=80)
        if val:
            fields.append({"field": "Signee Title", "text": " ".join(w[4] for w in val),
                            "bbox_px": union_bbox(val), "bbox_pt": to_pdf_coords(union_bbox(val))})

    # Document number in the footer, e.g. "DOC1098513/Rev9"
    for w in words:
        if w[4].startswith("DOC") and "/" in w[4]:
            fields.append({"field": "Document Number", "text": w[4],
                            "bbox_px": (w[0], w[1], w[2], w[3]), "bbox_pt": to_pdf_coords((w[0], w[1], w[2], w[3]))})
            break

    # The rotated approval stamp along the left margin -- notable because it's
    # recovered here by OCR even though it is NOT recoverable by native
    # get_text() extraction (it's not real text in the PDF, just pixels; OCR
    # only "sees" it because it OCRs the whole rendered page as an image).
    stamp_words = [w for w in words if "090177" in w[4] or w[4] in ("On:", "(GMT)")
                   or ("-2022" in w[4]) or (":" in w[4] and w[4][0].isdigit())]
    if stamp_words:
        fields.append({
            "field": "Rotated Approval Stamp (recovered via OCR only)",
            "text": " ".join(w[4] for w in stamp_words),
            "bbox_px": union_bbox(stamp_words), "bbox_pt": to_pdf_coords(union_bbox(stamp_words)),
        })

    return fields


def extract_table(words):
    # Exclude the rotated left-margin stamp column (x0 < 150px) -- it's
    # unrelated to the table and otherwise gets clustered into "rows" between
    # real table rows purely by y-coordinate coincidence.
    table_words = [w for w in words if w[0] >= 150]
    all_rows = cluster_rows(table_words)

    def is_header(row):
        texts = [w[4].lower() for w in row]
        return "product/part" in texts and any("code" in t for t in texts)

    header_idx = next((i for i, r in enumerate(all_rows) if is_header(r)), None)
    if header_idx is None:
        return None

    header_row = all_rows[header_idx]
    table = [("HEADER", header_row)]
    prev_y = header_row[0][1]
    for row in all_rows[header_idx + 1:]:
        if row[0][1] - prev_y > 100:  # big gap -> left the table area
            break
        table.append(("ROW", row))
        prev_y = row[0][1]
        if len(table) > 6:
            break
    return table




## Run OCR and extract fields

In [ ]:
img = render_page(doc, 0)
words = ocr_words(img)
print(f"OCR'd {len(words)} words")

fields = extract_fields(words)
for f in fields:
    print(f"{f['field']:45s} -> {f['text']!r:60s} bbox(pt)={f['bbox_pt']}")


## Extract the Product/Part table

In [ ]:
table = extract_table(words)
table_struct = []
for kind, row in table:
    texts = [w[4] for w in row]
    print(f"[{kind}] {texts}")
    table_struct.append({"row_type": kind, "cells": texts,
                          "bbox_px": union_bbox(row), "bbox_pt": to_pdf_coords(union_bbox(row))})


## Save structured JSON

In [ ]:
output = {"fields": fields, "product_part_table": table_struct}
with open("animal_origin_extraction_results.json", "w") as f:
    json.dump(output, f, indent=2)
print(json.dumps(fields, indent=2))


## Optional: draw OpenCV rectangles to visually confirm the boxes

In [ ]:
for f in fields:
    x0, y0, x1, y1 = f["bbox_px"]
    color = (0, 0, 255) if "Stamp" in f["field"] else (0, 128, 255)
    cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), color, 3)
    cv2.putText(img, f["field"], (int(x0), max(int(y0) - 8, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
for row in table_struct:
    x0, y0, x1, y1 = row["bbox_px"]
    cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), (0, 180, 0), 2)
cv2.imwrite("ocr_annotated_animal_origin.png", img)

from IPython.display import Image, display
display(Image(filename="ocr_annotated_animal_origin.png", width=650))


## Reflection

**Which fields were easy to extract?**
Every printed `Label: Value` field on this page — Regarding, EMA standard, Document Number, the
Product/Part table — came out clean once anchored on the label text.

**Did any get missed or show up incorrectly?**
One field was missed completely and it's worth calling out: **"Name: Susan Boucher"**, printed directly
under the "Signature:" line, got **zero OCR hits at any confidence level** — not a low-confidence guess,
just nothing. Cropping that exact region confirms the text is perfectly legible to a human; the most
likely explanation is that Tesseract's page-segmentation step merged that text block with the overlapping
handwritten signature stroke above it and discarded it as non-text. "Title" and "Date" on the same row,
which don't touch the signature, extracted fine.

**How clean was the table output?**
Clean, but only after filtering out the rotated stamp column. The stamp's OCR'd words sit at the far-left
margin (x≈55px) at *y-coordinates that happen to fall between real table rows*, so a naive row-clustering
pass (grouping purely by y-position) pulled stamp fragments into the table by coincidence. Restricting the
table search to the table's actual x-range fixed it immediately — a reminder that OCR row-reconstruction
needs both axes, not just y.

**Were the bounding boxes helpful or frustrating?**
Extremely helpful, and it produced this notebook's most interesting result: Tesseract's default OCR call
actually **recovered the rotated left-margin approval stamp** ("090177e19abdfceb\Approved\Approved On:
13-Jul-2022 20:47 (GMT)") as real text, with a very tall, narrow bounding box that visually confirms it read
the text sideways. That directly contradicts what we found in the earlier native-text-extraction exercise:
`get_text("words")` could never see this stamp because it isn't real text in the PDF, but Tesseract OCRs
whatever is visually on the page, orientation included, since it treats the whole page as pixels.

## Key date fields identified
- Signature Date: 19 March 2021
- Rotated stamp timestamp (recovered only via OCR): 13-Jul-2022 20:47 (GMT)

## What surprised me most
Two things pulled in opposite directions. First, OCR actually **beat** native text extraction on the
rotated stamp — something I'd written off as permanently unrecoverable in the earlier exercise turned out
to be readable after all, just not through `get_text()`. Second, OCR **lost** a piece of plain, legible,
non-rotated printed text ("Susan Boucher") for what looks like a segmentation reason tied to the nearby
signature scrawl, with no low-confidence trace to even flag it as uncertain. The lesson: OCR isn't strictly
better or worse than native extraction, it fails in a completely different, less predictable way, so the
only real safeguard is spot-checking extracted output against the rendered page rather than trusting either
method blindly.
